# Online Retail II: Raw Data Audit

This notebook documents the structure, completeness, consistency, and temporal coverage of the raw UCI Online Retail II workbook. The scope is strictly diagnostic: no rows are removed, no values are imputed, and no churn definition or modeling decision is introduced.

The workbook contains two worksheets that are audited separately until their temporal overlap and record consistency are understood.

In [1]:
import pandas as pd

workbook = pd.ExcelFile("../data/raw/online_retail_II.xlsx")
for sheet_name in workbook.sheet_names:
    headers = pd.read_excel(workbook, sheet_name=sheet_name, nrows=0)
    print(f"{sheet_name}: {headers.columns.tolist()}")

Year 2009-2010: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Year 2010-2011: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


## 1. Raw schema and column definitions

The workbook schema is checked before loading the complete dataset. The definitions below follow the UCI documentation while preserving the exact column names observed in this file.

| Raw column | Meaning |
| --- | --- |
| `Invoice` | Invoice or transaction identifier. UCI documentation notes that codes beginning with `C` represent cancellations. |
| `StockCode` | Product or item identifier. |
| `Description` | Product name or description. |
| `Quantity` | Number of units recorded for that product line. |
| `InvoiceDate` | Date and time when the transaction was generated. |
| `Price` | Unit price, in pounds sterling. |
| `Customer ID` | Customer identifier. |
| `Country` | Customer's country of residence. |

`Invoice`, `Price`, and `Customer ID` correspond to the UCI documentation's `InvoiceNo`, `UnitPrice`, and `CustomerID`, respectively.

## 2. Initial transaction preview

Five rows from each worksheet are displayed to confirm that values align with the observed schema. This preview is illustrative only and is not used to infer distributions, missingness, or data quality across the full dataset.

In [ ]:
for sheet_name in workbook.sheet_names:
    print(sheet_name)
    display(pd.read_excel(workbook, sheet_name=sheet_name, nrows=5))

## 3. Full workbook load

Each worksheet is loaded into a separate DataFrame so that its dimensions and quality can be assessed independently. The sheets are not concatenated because their temporal relationship has not yet been resolved.

In [ ]:
transactions_2009_2010 = pd.read_excel(workbook, sheet_name="Year 2009-2010")
transactions_2010_2011 = pd.read_excel(workbook, sheet_name="Year 2010-2011")
print("Year 2009-2010:", transactions_2009_2010.shape)
print("Year 2010-2011:", transactions_2010_2011.shape)

## 4. Inferred data types and completeness

Pandas' inferred data types and non-null counts are reviewed for each worksheet. Inferred storage types are not treated as semantic types: for example, `Customer ID` is an identifier even when pandas stores it as a numeric column.

In [ ]:
transactions_2009_2010.info()
transactions_2010_2011.info()

## 5. Missing-value profile

Missing-value counts and percentages are reported separately by worksheet. This step measures data availability only; no row exclusion, imputation, or customer-recovery rule is applied.

In [ ]:
for sheet_name, transactions in [
    ("Year 2009-2010", transactions_2009_2010),
    ("Year 2010-2011", transactions_2010_2011),
]:
    print(sheet_name)
    missing_values = pd.DataFrame({
        "Count": transactions.isna().sum(),
        "Percent": (transactions.isna().mean() * 100).round(2),
    })
    display(missing_values)

## 6. Exact duplicate count

Exact duplicates are rows with identical values across all eight raw columns. They are counted but not removed because the dataset has no invoice-line identifier and cannot distinguish an accidental copy from two legitimate identical line entries.

In [ ]:
print("Year 2009-2010:", transactions_2009_2010.duplicated().sum())
print("Year 2010-2011:", transactions_2010_2011.duplicated().sum())

### 6.1 Duplicate examples

The first ten rows involved in exact duplication are displayed for each worksheet. `keep=False` includes the initial occurrence and all repetitions; the DataFrame index is not part of the duplicate comparison.

In [ ]:
for sheet_name, transactions in [
    ("Year 2009-2010", transactions_2009_2010),
    ("Year 2010-2011", transactions_2010_2011),
]:
    print(sheet_name)
    display(transactions[transactions.duplicated(keep=False)].head(10))

## 7. Numerical integrity

`Quantity` and `Price` are the two numerical measures in the raw schema. Their summaries identify the central range and expose negative, zero, or extreme observations without assuming that unusual values are erroneous.

In [ ]:
display(transactions_2009_2010[["Quantity", "Price"]].describe())
display(transactions_2010_2011[["Quantity", "Price"]].describe())

### 7.1 Negative and zero values

Negative and zero values are counted separately for `Quantity` and `Price`. Their signs may reflect cancellations, returns, stock corrections, free items, or accounting adjustments, so they require semantic investigation before treatment.

In [ ]:
print("2009-2010 | Quantity < 0:", (transactions_2009_2010["Quantity"] < 0).sum())
print("2009-2010 | Quantity = 0:", (transactions_2009_2010["Quantity"] == 0).sum())
print("2009-2010 | Price < 0:", (transactions_2009_2010["Price"] < 0).sum())
print("2009-2010 | Price = 0:", (transactions_2009_2010["Price"] == 0).sum())
print("2010-2011 | Quantity < 0:", (transactions_2010_2011["Quantity"] < 0).sum())
print("2010-2011 | Quantity = 0:", (transactions_2010_2011["Quantity"] == 0).sum())
print("2010-2011 | Price < 0:", (transactions_2010_2011["Price"] < 0).sum())
print("2010-2011 | Price = 0:", (transactions_2010_2011["Price"] == 0).sum())

## 8. Cancellation and adjustment patterns

UCI documents invoices beginning with `C` as cancellations. A cross-tabulation tests how consistently this convention aligns with negative quantities and distinguishes documented cancellations from other negative-quantity operations.

In [ ]:
print("Year 2009-2010")
display(pd.crosstab(transactions_2009_2010["Invoice"].astype(str).str.startswith("C"), transactions_2009_2010["Quantity"] < 0, margins=True))
print("Year 2010-2011")
display(pd.crosstab(transactions_2010_2011["Invoice"].astype(str).str.startswith("C"), transactions_2010_2011["Quantity"] < 0, margins=True))

### 8.1 Cancellation exceptions and non-cancellation negatives

Invoices beginning with `C` but having non-negative quantities are inspected alongside negative-quantity rows whose invoice does not begin with `C`. Descriptions such as stock-loss or damage notes may indicate internal adjustments rather than customer returns; records remain unclassified and unchanged.

In [51]:
columns = ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "Country"]
for sheet_name, transactions in [("Year 2009-2010", transactions_2009_2010), ("Year 2010-2011", transactions_2010_2011)]:
    invoice_starts_c = transactions["Invoice"].astype(str).str.startswith("C")
    print(f"{sheet_name}: C invoice with non-negative quantity")
    display(transactions.loc[invoice_starts_c & (transactions["Quantity"] >= 0), columns])
    print(f"{sheet_name}: non-C invoice with negative quantity")
    display(transactions.loc[~invoice_starts_c & (transactions["Quantity"] < 0), columns].head(20))

Year 2009-2010: C invoice with non-negative quantity


,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
76799,C496350,M,Manual,1,373.57,NaN,United Kingdom


Year 2009-2010: non-C invoice with negative quantity


,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,0.0,NaN,United Kingdom
283,489463,71477,short,-240,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,0.0,NaN,United Kingdom
4538,489820,21133,invcd as 84879?,-720,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,0.0,NaN,United Kingdom


Year 2010-2011: C invoice with non-negative quantity


,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country


Year 2010-2011: non-C invoice with negative quantity


,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
2406,536589,21777,NaN,-10,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,0.0,NaN,United Kingdom
7192,537000,21414,NaN,-22,0.0,NaN,United Kingdom
7193,537001,21653,NaN,-6,0.0,NaN,United Kingdom
7195,537003,85126,NaN,-2,0.0,NaN,United Kingdom
7196,537004,21814,NaN,-30,0.0,NaN,United Kingdom
7197,537005,21692,NaN,-70,0.0,NaN,United Kingdom


## 9. Price anomalies

All negative-price rows are displayed because they are rare and can materially affect future financial calculations. Their descriptions are used to determine whether they resemble product sales or accounting adjustments; no statistical outlier treatment is applied.

In [ ]:
display(transactions_2009_2010[transactions_2009_2010["Price"] < 0])
display(transactions_2010_2011[transactions_2010_2011["Price"] < 0])

### 9.1 Zero-price descriptions

The most frequent descriptions among zero-price rows are reviewed to separate customer-facing products from incomplete or operational records. Descriptions provide context but do not, by themselves, establish why a price is zero.

In [50]:
print("Year 2009-2010")
display(transactions_2009_2010.loc[transactions_2009_2010["Price"] == 0, "Description"].value_counts(dropna=False).head(15))
print("Year 2010-2011")
display(transactions_2010_2011.loc[transactions_2010_2011["Price"] == 0, "Description"].value_counts(dropna=False).head(15))

Year 2009-2010


Description
NaN                                    2928
?                                        45
damages                                  39
damaged                                  38
missing                                  24
checked                                   8
OWL DOORSTOP                              8
POLYESTER FILLER PAD 45x45cm              7
PICNIC BASKET WICKER LARGE                7
given away                                6
AIRLINE BAG VINTAGE WORLD CHAMPION        6
dotcom                                    6
HEART OF WICKER SMALL                     6
FLAG OF ST GEORGE CAR FLAG                6
SMALL POPCORN HOLDER                      5
Name: count, dtype: int64

Year 2010-2011


Description
NaN                              1454
check                             159
?                                  47
damages                            45
damaged                            43
found                              25
sold as set on dotcom              20
adjustment                         16
Damaged                            14
Unsaleable, destroyed.              9
thrown away                         9
FRENCH BLUE METAL DOOR SIGN 1       9
FRENCH BLUE METAL DOOR SIGN 8       8
amazon                              8
Found                               8
Name: count, dtype: int64

### 9.2 Zero-price rows and customer identification

Zero-price rows are divided by customer-ID availability. A valid identifier establishes that the line is customer-linked, but it does not generate revenue and does not yet determine whether the event should reset a future recency measure.

In [ ]:
print("2009-2010 | Valid Customer ID:", transactions_2009_2010.loc[(transactions_2009_2010["Price"] == 0) & (transactions_2009_2010["Customer ID"].notna())].shape[0])
print("2009-2010 | Missing Customer ID:", transactions_2009_2010.loc[(transactions_2009_2010["Price"] == 0) & (transactions_2009_2010["Customer ID"].isna())].shape[0])
print("2010-2011 | Valid Customer ID:", transactions_2010_2011.loc[(transactions_2010_2011["Price"] == 0) & (transactions_2010_2011["Customer ID"].notna())].shape[0])
print("2010-2011 | Missing Customer ID:", transactions_2010_2011.loc[(transactions_2010_2011["Price"] == 0) & (transactions_2010_2011["Customer ID"].isna())].shape[0])

## 10. Temporal coverage

The earliest and latest `InvoiceDate` in each worksheet define the observed data horizon. These boundaries are verified directly rather than inferred from worksheet names.

In [58]:
display(transactions_2009_2010["InvoiceDate"].agg(["min", "max"]))
display(transactions_2010_2011["InvoiceDate"].agg(["min", "max"]))

min   2009-12-01 07:45:00
max   2010-12-09 20:01:00
Name: InvoiceDate, dtype: datetime64[ns]

min   2010-12-01 08:26:00
max   2011-12-09 12:50:00
Name: InvoiceDate, dtype: datetime64[ns]

### 10.1 Overlap between worksheets

The common time interval is extracted from both worksheets. Exact matches across all eight columns are counted after temporary within-sheet deduplication to test whether a direct concatenation would duplicate transactions. This diagnostic does not define the final reconciliation rule.

In [57]:
overlap_start = max(transactions_2009_2010["InvoiceDate"].min(), transactions_2010_2011["InvoiceDate"].min())
overlap_end = min(transactions_2009_2010["InvoiceDate"].max(), transactions_2010_2011["InvoiceDate"].max())
overlap_2009_2010 = transactions_2009_2010[transactions_2009_2010["InvoiceDate"].between(overlap_start, overlap_end)]
overlap_2010_2011 = transactions_2010_2011[transactions_2010_2011["InvoiceDate"].between(overlap_start, overlap_end)]
shared_rows = overlap_2009_2010.drop_duplicates().merge(overlap_2010_2011.drop_duplicates())
print("Overlap period:", overlap_start, "to", overlap_end)
print("Rows in Year 2009-2010:", len(overlap_2009_2010))
print("Rows in Year 2010-2011:", len(overlap_2010_2011))
print("Unique exact rows shared:", len(shared_rows))

Overlap period: 2010-12-01 08:26:00 to 2010-12-09 20:01:00
Rows in Year 2009-2010: 22523
Rows in Year 2010-2011: 22523
Unique exact rows shared: 22202


## 11. Column cardinality

The number of distinct non-missing values is reported for every column in each worksheet. Identifier cardinalities describe the scale of invoices, products, customers, and countries; cardinalities of numerical measures describe only the number of observed values.

In [ ]:

transactions_2009_2010.apply(pd.Series.nunique)


Invoice        28816
StockCode       4632
Description     4681
Quantity         825
InvoiceDate    25296
Price           1606
Customer ID     4383
Country           40
dtype: int64

In [62]:
transactions_2010_2011.apply(pd.Series.nunique)


Invoice        25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
Price           1630
Customer ID     4372
Country           38
dtype: int64

### 11.1 Cardinality observations

The 2009–2010 worksheet contains 4,632 distinct `StockCode` values, compared with 4,070 in 2010–2011. This difference does not, by itself, demonstrate product introductions or removals because special operational codes and inconsistent product coverage may contribute to the counts.

The worksheets contain 4,383 and 4,372 identified customers, respectively. These counts exclude missing IDs and cannot be added because customers may appear in both periods and the worksheets overlap temporally.

## 12. Invoice-level consistency

A valid invoice should normally be associated with one customer, one timestamp, and one country. For each worksheet, invoices with multiple distinct non-missing values are counted for these three fields. Missing customer IDs are not treated as conflicting customer identities, and no records are modified.

In [ ]:
for sheet_name, transactions in [
    ("Year 2009-2010", transactions_2009_2010),
    ("Year 2010-2011", transactions_2010_2011),
]:
    invoice_consistency = transactions.groupby("Invoice").agg({
        "Customer ID": "nunique",
        "InvoiceDate": "nunique",
        "Country": "nunique",
    })
    print(sheet_name)
    display((invoice_consistency > 1).sum().rename("Inconsistent invoices"))

## 13. Geographic coverage

Transaction lines, unique invoices, and identified customers are counted by country for each worksheet. This describes geographic concentration without restricting the analytical population or interpreting revenue differences. Missing customer IDs are excluded only from the customer count.

In [ ]:
for sheet_name, transactions in [
    ("Year 2009-2010", transactions_2009_2010),
    ("Year 2010-2011", transactions_2010_2011),
]:
    country_summary = transactions.groupby("Country").agg(
        Rows=("Invoice", "size"),
        Invoices=("Invoice", "nunique"),
        Customers=("Customer ID", "nunique"),
    ).sort_values("Rows", ascending=False)
    print(sheet_name)
    display(country_summary)

## 14. Potential recovery of missing customer IDs

Invoices are classified by the number of known customer IDs and missing customer-ID rows they contain. A missing line is considered potentially recoverable only when its invoice contains exactly one known customer. This section measures recovery potential but does not fill any identifier.

In [ ]:
for sheet_name, transactions in [
    ("Year 2009-2010", transactions_2009_2010),
    ("Year 2010-2011", transactions_2010_2011),
]:
    invoice_customer_ids = transactions.groupby("Invoice")["Customer ID"].agg(
        Known_customers="nunique",
        Missing_rows=lambda customer_ids: customer_ids.isna().sum(),
    )
    recoverable = (invoice_customer_ids["Known_customers"] == 1) & (invoice_customer_ids["Missing_rows"] > 0)
    print(sheet_name)
    print("Fully anonymous invoices:", ((invoice_customer_ids["Known_customers"] == 0) & (invoice_customer_ids["Missing_rows"] > 0)).sum())
    print("Potentially recoverable invoices:", recoverable.sum())
    print("Potentially recoverable rows:", invoice_customer_ids.loc[recoverable, "Missing_rows"].sum())
    print("Ambiguous invoices:", ((invoice_customer_ids["Known_customers"] > 1) & (invoice_customer_ids["Missing_rows"] > 0)).sum())

## 15. Monthly temporal continuity

Transaction lines and unique invoices are counted by calendar month for each worksheet. This check identifies missing, incomplete, or unusually active periods without merging the worksheets or defining anomaly thresholds.

In [ ]:
for sheet_name, transactions in [
    ("Year 2009-2010", transactions_2009_2010),
    ("Year 2010-2011", transactions_2010_2011),
]:
    monthly_summary = transactions.groupby(transactions["InvoiceDate"].dt.to_period("M")).agg(
        Rows=("Invoice", "size"),
        Invoices=("Invoice", "nunique"),
    )
    print(sheet_name)
    display(monthly_summary)

### 15.1 Feature hypothesis for later: customer seasonality

Monthly activity may reveal customers whose purchases are concentrated around recurring occasions or seasonal periods. Such customers are not necessarily more likely to churn: a long off-season gap may be normal behavior and could otherwise be mislabeled as inactivity. Seasonality may therefore help distinguish expected purchase cycles from genuine disengagement.

A simple cutoff-month variable is not equivalent to customer seasonality. Within a single snapshot, every customer shares the same cutoff month, so the variable provides no cross-sectional information. A customer-level seasonal profile would instead need to be calculated only from transactions observed before the cutoff, using an explicitly approved definition and historical window.

This is a feature hypothesis for later discussion in the feature-engineering stage. No seasonal feature, holiday calendar, or behavioral rule is created in this audit.

## 16. Product-code and description consistency

The raw relationship between `StockCode` and `Description` is tested in both directions. Counts identify product codes linked to multiple non-missing descriptions and descriptions linked to multiple product codes. No text normalization, code correction, or product consolidation is applied.

In [ ]:
for sheet_name, transactions in [
    ("Year 2009-2010", transactions_2009_2010),
    ("Year 2010-2011", transactions_2010_2011),
]:
    descriptions_per_code = transactions.groupby("StockCode")["Description"].nunique()
    codes_per_description = transactions.groupby("Description")["StockCode"].nunique()
    print(sheet_name)
    print("StockCodes with multiple descriptions:", (descriptions_per_code > 1).sum())
    print("Descriptions with multiple StockCodes:", (codes_per_description > 1).sum())

### 16.1 Product codes with the most descriptions

The ten `StockCode` values with the largest number of distinct non-missing descriptions are selected for each worksheet. Each distinct `StockCode`–`Description` pair is then displayed on a separate row so that raw labels remain fully readable.

In [90]:
for sheet_name, transactions in [
    ("Year 2009-2010", transactions_2009_2010),
    ("Year 2010-2011", transactions_2010_2011),
]:
    description_counts = transactions.groupby("StockCode")["Description"].nunique()
    top_codes = description_counts.sort_values(ascending=False).head(10).index
    description_examples = transactions.loc[
        transactions["StockCode"].isin(top_codes), ["StockCode", "Description"]
    ].dropna().drop_duplicates().sort_values(["StockCode", "Description"])
    print(sheet_name)
    display(description_examples)

Year 2009-2010


,nunique,unique
StockCode,,
22423,6,"[REGENCY CAKESTAND 3 TIER, smashed, damaged, b..."
22734,5,"[SET OF 6 RIBBONS VINTAGE CHRISTMAS, Carton qn..."
22345,4,"[PARTY PIZZA DISH BLUE+WHITE SPOT , PARTY PIZZ..."
85099B,4,"[JUMBO BAG RED WHITE SPOTTY , RED RETROSPOT JU..."
22384,4,"[LUNCHBAG PINK RETROSPOT, LUNCH BAG PINK RETRO..."
47566B,4,"[TEA TIME PARTY BUNTING, missing, correct prev..."
21843,4,"[RETRO SPOT CAKE STAND, RED RETROSPOT CAKE STA..."
48173C,4,"[DOOR MAT BLACK FLOCK , MIA, DOORMAT BLACK FLO..."
84016,4,"[FLAG OF ST GEORGE CAR FLAG, ebay sales, invoi..."


Year 2010-2011


,nunique,unique
StockCode,,
20713,8,"[JUMBO BAG OWLS, wrongly marked. 23343 in box,..."
23084,7,"[RABBIT NIGHT LIGHT, temp adjustment, allocate..."
85175,6,"[CACTI T-LIGHT CANDLES, dotcom sold sets, Amaz..."
21830,6,"[ASSORTED CREEPY CRAWLIES, MERCHANT CHANDLER C..."
21181,5,"[PLEASE ONE PERSON METAL SIGN, on cargo order,..."
85172,5,"[HYACINTH BULB T-LIGHT CANDLES, Dotcom set, wr..."
23131,5,"[MISTLETOE HEART WREATH CREAM, MISELTOE HEART ..."
72807A,5,"[SET/3 ROSE CANDLE IN JEWELLED BOX, wet pallet..."
23343,5,"[JUMBO BAG VINTAGE CHRISTMAS , came coded as 2..."


### 16.2 Sample of operational descriptions

A small list of exact operational descriptions is used to inspect their invoice, quantity, price, and customer identifier. This representative sample avoids matching ordinary product names and is not used to classify or remove records.

In [ ]:
operational_descriptions = [
    "damages", "damaged", "broken", "faulty", "smashed",
    "missing", "amendment", "adjustment", "found", "wet",
]
for sheet_name, transactions in [
    ("Year 2009-2010", transactions_2009_2010),
    ("Year 2010-2011", transactions_2010_2011),
]:
    operational_rows = transactions["Description"].str.lower().isin(operational_descriptions)
    print(sheet_name)
    display(transactions.loc[operational_rows, ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]].head(30).reset_index(drop=True))

## 17. Audit summary

### Dataset structure

- Both worksheets contain the same eight columns and consistent pandas data types.
- `Year 2009-2010` contains 525,461 rows and 28,816 distinct invoices.
- `Year 2010-2011` contains 541,910 rows and 25,900 distinct invoices.
- A row represents an invoice line rather than a complete invoice, so one invoice can contain multiple products.
- `StockCode` is the primary product identifier. `Description` is not a stable identifier because it also contains inconsistent labels and operational notes.

### Data-quality findings

- Missing values are concentrated in `Customer ID` and `Description`; no imputation or row exclusion has been applied.
- The worksheets contain 6,865 and 5,268 exact duplicate occurrences. Their status remains unresolved because the dataset has no invoice-line identifier.
- Negative quantities occur in 12,326 and 10,624 rows. They include documented cancellation invoices beginning with `C` as well as non-customer stock or operational adjustments.
- In the inspected operational-description sample, damaged, missing, wet, or smashed items appear as negative inventory movements, while items described as `found` appear as positive inventory movements. The displayed records have a zero price and no customer ID, which supports interpreting them as internal stock adjustments rather than customer transactions.
- No zero quantities were observed.
- Negative prices occur in only three and two rows, all described as bad-debt adjustments rather than ordinary product sales.
- Zero prices occur in 3,687 and 2,515 rows. Only 31 and 40 of those rows, respectively, have a valid customer ID; customer-linked zero-price items are retained as activity with zero direct revenue, while their future recency treatment remains undecided.

### Invoice and customer integrity

- No invoice is associated with multiple known customers or multiple countries.
- Forty-one and 43 invoice identifiers contain more than one timestamp. The cause has not been established, but the issue affects a small share of invoices.
- The worksheets contain 4,383 and 4,372 identified customers. These counts cannot be added because customers may occur in both periods.
- There are 5,229 and 3,710 fully anonymous invoices. No invoice mixes known and missing customer IDs, so invoice-level propagation cannot recover any missing customer identity.
- Anonymous invoices cannot support customer-level recency, frequency, churn labels, or targeted retention actions.

### Temporal and geographic coverage

- The observed period runs from 1 December 2009 to 9 December 2011.
- The worksheets overlap from 1 December 2010 at 08:26 to 9 December 2010 at 20:01. Each contains 22,523 rows during this interval, with 22,202 unique exact records shared across the two sheets. A reconciliation rule is required before concatenation.
- The United Kingdom dominates the first worksheet with 485,852 of 525,461 rows, approximately 92.5%. Restricting the population to the UK has not been approved.
- A future binary `is_uk` feature is a candidate hypothesis, not an audited feature or a demonstrated churn signal.
- Customer seasonality is also a candidate hypothesis. It may distinguish normal off-season inactivity from genuine disengagement, but no seasonal definition or feature has been approved.

### Decisions explicitly deferred

- Treatment of exact duplicates within each worksheet.
- Reconciliation of the duplicated temporal overlap.
- Formal classification of cancellations, returns, stock adjustments, and accounting records.
- Final inclusion rules for anonymous invoices and zero-price customer activity.
- Product-code normalization and treatment of administrative stock codes.
- Churn definition, inactivity threshold, temporal snapshots, feature windows, and modeling population.

No cleaning, feature engineering, churn labeling, or temporal splitting has been performed in this notebook.